# TP2 : topics et messages

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 11:50-12:30, après la démo du TP (11:45-11:50) |
| Compétences | C4 Administrer les topics ; C5 Produire et consommer |
| Slides de rappel | S6 (topics et partitions), S7 (offsets et ordre), S8 (partitionnement avec et sans clé), S22 (outils de `bin/`) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma qu'au TP1 : une explication, un **exemple en ligne de commande**, une **zone `# TODO`** à compléter en Python, puis une **vérification**.
- Quand la commande modifie le cluster (création d'un topic, envoi de messages), son équivalent en ligne de commande est affiché **sans être exécuté**, avec la mention « ne pas lancer en plus du TODO » : le lancer en plus doublerait l'opération et fausserait les vérifications.
- Les appels du client d'administration (`admin = AdminClient(CONF)`, créé dans chaque cellule qui en a besoin) renvoient un dictionnaire de **futurs** : `{nom: futur}`. `futur.result()` attend la réponse du cluster et lève `KafkaException` en cas d'échec ; `e.args[0].name()` donne alors le code d'erreur.
- Ce TP utilise aussi le producteur (`Producer`) et le consommateur (`Consumer`) de `confluent-kafka`.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, ecrites ici : elles ne font pas partie de confluent-kafka
def attendre(condition, timeout=60, pas=2, message="condition"):
    """Rappelle condition() jusqu'a obtenir une valeur vraie, et la renvoie."""
    debut = time.time()
    while True:
        try:
            valeur = condition()
        except Exception:
            valeur = None
        if valeur:
            return valeur
        if time.time() - debut >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(pas)

def tableau(lignes, entetes):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in ligne) + " |" for ligne in lignes]
    display(Markdown("\n".join(md)))

In [ ]:
import re

from confluent_kafka import Consumer, KafkaException, Producer, TopicCollection, TopicPartition
from confluent_kafka.admin import (AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource,
                                   NewPartitions, NewTopic, OffsetSpec)

print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- constater que la création automatique des topics est désactivée, et pourquoi ;
- créer, décrire, modifier et supprimer un topic ;
- lire le placement des répliques : leader, répliques, ISR ;
- produire des messages avec clé depuis un fichier, et sans clé ;
- lire une partition à partir d'un offset donné ;
- vérifier que tous les messages d'une même clé arrivent dans la même partition ;
- mesurer l'effet d'un ajout de partitions sur la répartition des clés ;
- poser, lire et retirer une configuration de topic.

## 2. Prérequis

- TP1 terminé : lab démarré et `scripts/validate.sh` au vert.
- Jeu de données `lab/data/orders.txt` : 30 commandes au format `client|commande JSON`, clients `C001` à `C007`. Il est lisible depuis le notebook (`"data/orders.txt"`) et depuis `tools` (`/opt/lab/data/orders.txt`).

## 3. Architecture utilisée

```
  jupyter (client Python) et tools (CLI) ──> kafka-1, kafka-2, kafka-3
                                             (listener INTERNAL, port 19092, BS)

  Topic orders, 3 partitions, RF 3 : chaque partition a 1 leader et 2 followers,
  répartis sur les 3 brokers.

            kafka-1          kafka-2          kafka-3
  orders-0  follower         LEADER           follower
  orders-1  follower         follower         LEADER
  orders-2  LEADER           follower         follower
```

Le placement exact des leaders varie d'un poste à l'autre.

## 4. Contexte métier

Le service Commandes de NovaShop va publier chaque commande dans un topic `orders`. Exigence métier : les commandes d'un même client doivent être traitées dans l'ordre. L'équipe Data veut aussi un topic `clickstream` pour la navigation, conservé 24 heures.

Vous préparez ces topics et vérifiez que la clé de partitionnement garantit bien l'ordre par client. La fiche des topics affichée en fin de notebook résume ce que vous avez livré.

## 5. Énoncé

1. Constatez que produire vers un topic qui n'existe pas échoue.
2. Créez `orders` (3 partitions, facteur de réplication 3) et alimentez-le avec `orders.txt`, en utilisant l'identifiant client comme clé.
3. Prouvez que toutes les commandes d'un même client sont dans la même partition.
4. Passez `orders` à 6 partitions, alimentez-le à nouveau et mesurez ce que l'opération change pour l'ordre par client.
5. Créez `clickstream` (6 partitions) avec une rétention de 24 heures.

## 6. Étapes

### 6.1 Produire vers un topic absent

Le lab fixe `auto.create.topics.enable=false` sur chaque nœud : un producteur ne peut pas créer un topic en y écrivant. On le constate avec l'outil Java, puis avec le client Python.

**Exemple en ligne de commande** (le producteur abandonne au bout de 5 s) :

In [ ]:
!grep auto.create config/kafka/kafka-1/server.properties
!docker exec kafka-tools bash -c "echo 'test' | kafka-console-producer.sh --bootstrap-server $BS --topic orders --command-property max.block.ms=5000"

In [ ]:
# TODO 6.1 : produire vers le topic absent avec le client Python
# Ce producteur declare un topic inconnu au bout de 5 s (30 s par defaut).
producteur_test = Producer({"bootstrap.servers": BS, "topic.metadata.propagation.max.ms": 5000, "log_level": 4})
erreurs = []

# 1. Envoyez le message "test" vers "orders" :
#      producteur_test.produce(topic, value=..., on_delivery=rappel)
#    rappel est une fonction rappel(err, msg), appelee une fois par message :
#    faites-lui ajouter err a la liste erreurs (err vaut None si le message est ecrit).
# 2. Attendez le rapport de livraison : producteur_test.flush(15).

print(erreurs)

In [ ]:
# Verification 6.1
admin = AdminClient(CONF)
assert len(erreurs) == 1 and erreurs[0] is not None, "A revoir : un rapport de livraison, en erreur"
print("OK  un rapport de livraison, en erreur")
assert erreurs[0].name() == "UNKNOWN_TOPIC_OR_PART", "A revoir : code d'erreur : %s" % erreurs[0].name()
print("OK  code d'erreur : %s" % erreurs[0].name())
assert admin.list_topics("orders", timeout=10).topics["orders"].error is not None, "A revoir : le topic orders n'a pas ete cree par l'envoi"
print("OK  le topic orders n'a pas ete cree par l'envoi")

**Résultat attendu** :

- en ligne de commande, des avertissements `{orders=UNKNOWN_TOPIC_OR_PARTITION}`, puis au bout de 5 s `TimeoutException: Topic orders not present in metadata after 5000 ms.` ;
- en Python, au bout de 5 s, un rapport de livraison en erreur : `KafkaError{code=UNKNOWN_TOPIC_OR_PART,val=3,str="Broker: Unknown topic or partition"}`.

### 6.2 Créer le topic `orders`

Un topic se crée avec un nombre de partitions et un facteur de réplication (RF). Le RF ne peut pas dépasser le nombre de brokers.

**Exemple en ligne de commande** : demande d'un RF de 4 sur un cluster de 3 brokers.

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic orders-rf4 --partitions 1 --replication-factor 4

Équivalent en ligne de commande de la création de `orders` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic orders --partitions 3 --replication-factor 3
```

In [ ]:
# TODO 6.2 : creer le topic orders (3 partitions, facteur de replication 3)
# admin.create_topics([NewTopic(nom, num_partitions=..., replication_factor=...)])
# renvoie un dictionnaire {nom: futur} ; futur.result() attend la creation.
#   1. Creez orders et attendez le resultat.
#   2. Demandez orders-rf4 (1 partition, RF 4) avec l'option validate_only=True :
#      le cluster valide la demande sans rien creer. Rattrapez KafkaException
#      (try / except KafkaException as e) et rangez e.args[0].name() dans code_rf4.
code_rf4 = None

print("code_rf4 :", code_rf4)

In [ ]:
# Verification 6.2
admin = AdminClient(CONF)
meta_orders = admin.list_topics("orders", timeout=10).topics["orders"]
assert meta_orders.error is None, "A revoir : topic orders cree"
print("OK  topic orders cree")
assert len(meta_orders.partitions) == 3, "A revoir : 3 partitions"
print("OK  3 partitions")
assert code_rf4 == "INVALID_REPLICATION_FACTOR", "A revoir : RF 4 refuse : %s" % code_rf4
print("OK  RF 4 refuse : %s" % code_rf4)

**Résultat attendu** : `orders` est créé. Le facteur de réplication de 4 est refusé, en ligne de commande comme en Python :

```
Unable to replicate the partition 4 time(s): The target replication factor of 4 cannot be reached because only 3 broker(s) are registered or some brokers have all their log directories cordoned.
```

### 6.3 Décrire le topic

Chaque partition a un leader, une liste de répliques et un ISR (répliques synchronisées). La première réplique de la liste est le **leader préféré**.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic orders

In [ ]:
# TODO 6.3 : relever le placement des repliques de orders
# admin.describe_topics(TopicCollection(["orders"])) renvoie {nom: futur}.
# Le resultat a un attribut partitions : une liste d'objets avec les attributs
# id, leader (un noeud), replicas et isr (listes de noeuds). Un noeud a un attribut id.
# Construisez placement : liste de tuples (partition, leader, [repliques], [isr]),
# avec les identifiants des noeuds.
placement = []

tableau(placement, ["partition", "leader", "repliques", "isr"])

In [ ]:
# Verification 6.3
assert len(placement) == 3, "A revoir : 3 partitions decrites"
print("OK  3 partitions decrites")
assert all(len(rep) == 3 and set(isr) == set(rep) for _, _, rep, isr in placement), "A revoir : 3 repliques par partition, toutes dans l'ISR"
print("OK  3 repliques par partition, toutes dans l'ISR")
assert all(leader == rep[0] for _, leader, rep, _ in placement), "A revoir : chaque leader est le leader prefere"
print("OK  chaque leader est le leader prefere")
assert sorted(p[1] for p in placement) == [1, 2, 3], "A revoir : leaders repartis sur les 3 brokers"
print("OK  leaders repartis sur les 3 brokers")

**Résultat attendu** : 3 partitions, chacune avec 3 répliques toutes dans l'ISR, et des leaders répartis sur les 3 brokers. En ligne de commande, la ligne de titre affiche aussi des configurations : `min.insync.replicas=1`, `segment.bytes=1073741824`, `unclean.leader.election.enable=false`.

### 6.4 Produire avec clé

Chaque ligne de `orders.txt` est de la forme `client|commande JSON`. L'identifiant client sert de clé : le producteur calcule la partition à partir d'un hachage de la clé, donc toutes les commandes d'un client vont dans la même partition.

**Attention au partitionneur.** Les clients Java, dont les outils `kafka-*.sh`, utilisent le hachage murmur2. librdkafka, la bibliothèque du client Python, utilise par défaut un autre hachage (CRC32, partitionneur `consistent_random`) : la même clé irait dans une autre partition. Le réglage `partitioner=murmur2_random` aligne le client Python sur Java. Il est indispensable dès que des producteurs écrits dans plusieurs langages alimentent le même topic.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools head -3 /opt/lab/data/orders.txt

Équivalent en ligne de commande de l'envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|' \
  < /opt/lab/data/orders.txt
```

**Exécutez la cellule `TODO` une seule fois** : chaque exécution renvoie les 30 commandes. En cas de doublon, voir le point 12, erreurs fréquentes.

In [ ]:
# TODO 6.4 : envoyer les 30 commandes de orders.txt, avec le client comme cle
producteur = Producer({"bootstrap.servers": BS, "partitioner": "murmur2_random", "log_level": 4})


def produire_commandes():
    """Envoie orders.txt dans orders et renvoie le nombre de messages confirmes."""
    confirmes = []
    # 1. Lisez le fichier "data/orders.txt" ligne par ligne.
    # 2. Coupez chaque ligne au premier "|" : ligne.split("|", 1) donne [cle, valeur].
    # 3. producteur.produce("orders", key=cle, value=valeur, on_delivery=rappel)
    #    rappel(err, msg) ajoute msg a confirmes si err vaut None.
    # 4. producteur.flush(30) attend la confirmation de tous les messages.
    return len(confirmes)


envoyes = produire_commandes()
print(envoyes, "messages confirmes")

In [ ]:
# Verification 6.4
assert envoyes == 30, "A revoir : 30 messages confirmes par le cluster (%s)" % envoyes
print("OK  30 messages confirmes par le cluster (%s)" % envoyes)
fins_cli = !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic orders 2>&1
fins_cli = fins_cli.n
total = sum(int(n) for n in re.findall(r"^orders:\d+:(\d+)$", fins_cli, re.M))
assert total == 30, "A revoir : 30 messages dans orders (%d) : envoi fait une seule fois" % total
print("OK  30 messages dans orders (%d) : envoi fait une seule fois" % total)

**Résultat attendu** : `30 messages confirmes`. La première ligne du fichier est `C002|{"order_id":"O-1001","customer":"C002","amount":15.03,"city":"Lille"}`.

### 6.5 Lire les offsets et consommer

L'offset de fin d'une partition est l'offset du prochain message à écrire. Tant que rien n'a été supprimé, il est égal au nombre de messages de la partition.

Pour lire une partition précise à partir d'un offset, le consommateur utilise `assign()` : il lit directement, sans groupe actif ni rééquilibrage (les groupes sont le sujet du TP3).

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic orders
!docker exec kafka-tools kafka-console-consumer.sh --bootstrap-server $BS --topic orders --partition 0 --offset 0 --max-messages 5 --timeout-ms 10000 --formatter-property print.key=true --formatter-property print.partition=true --formatter-property print.offset=true

In [ ]:
# TODO 6.5 : offsets de fin, puis lecture des 5 premiers messages de la partition 0
# 1. Ecrivez offsets_de_fin(topic) : dictionnaire {partition: offset de fin}.
#    - nombre de partitions : len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
#    - admin.list_offsets({TopicPartition(topic, p): OffsetSpec.latest() for p in ...})
#      renvoie {TopicPartition: futur} ; le resultat a un attribut offset,
#      la cle TopicPartition un attribut partition.
def offsets_de_fin(topic):
    return {}


# 2. Completez lire(topic, partition, debut, nombre) : au plus nombre messages, a partir de l'offset debut.
#    consommateur.consume(n, timeout=10) renvoie une liste d'au plus n messages, vide si
#    rien n'arrive pendant le delai. Recommencez tant qu'il en manque ; arretez sur une liste vide.
def lire(topic, partition, debut, nombre):
    # Sans commit d'offsets, le groupe tp2-lecture n'est pas enregistre sur le cluster
    consommateur = Consumer({"bootstrap.servers": BS, "group.id": "tp2-lecture",
                             "enable.auto.commit": False, "log_level": 4})
    consommateur.assign([TopicPartition(topic, partition, debut)])
    messages = []
    # a completer
    consommateur.close()
    return messages


fins = offsets_de_fin("orders")
# Chaque message a les methodes partition(), offset(), key() et value() (en octets : .decode())
premiers = [(m.partition(), m.offset(), m.key().decode()) for m in lire("orders", 0, 0, 5)]

print(fins)
tableau(premiers, ["partition", "offset", "cle"])

In [ ]:
# Verification 6.5
assert sum(fins.values()) == 30, "A revoir : 30 messages au total : %s" % fins
print("OK  30 messages au total : %s" % fins)
assert fins == {0: 6, 1: 8, 2: 16}, "A revoir : repartition identique a celle du producteur Java : %s" % fins
print("OK  repartition identique a celle du producteur Java : %s" % fins)
assert len(premiers) == 5 and {p[2] for p in premiers} == {"C002"}, "A revoir : 5 premiers messages de la partition 0, tous du client C002"
print("OK  5 premiers messages de la partition 0, tous du client C002")

**Résultat attendu** : `{0: 6, 1: 8, 2: 16}`, identique à `kafka-get-offsets.sh` (`orders:0:6`, `orders:1:8`, `orders:2:16`). Les 5 premiers messages de la partition 0 sont tous du client `C002`, aux offsets 0 à 4.

### 6.6 Défi : correspondance clé et partition

Prouvez que toutes les commandes d'un même client sont dans une seule partition.

**Exemple en ligne de commande** : le consommateur affiche la partition et la clé de chaque message. En ligne de commande, `awk` et `sort -u` feraient le reste ; ici, ce sera Python.

In [ ]:
!docker exec kafka-tools bash -c "kafka-console-consumer.sh --bootstrap-server $BS --topic orders --from-beginning --max-messages 5 --timeout-ms 10000 --formatter-property print.key=true --formatter-property print.partition=true 2>/dev/null"

In [ ]:
# TODO 6.6 (defi) : pour chaque client, la ou les partitions de ses commandes
# Ecrivez carte_des_cles(topic), qui renvoie un dictionnaire {cle: [partitions triees]}.
# Lisez chaque partition en entier : de l'offset 0 a son offset de fin.
# Reutilisez offsets_de_fin() et lire() de l'etape 7.5.
def carte_des_cles(topic):
    carte = {}
    # a completer
    return carte


carte = carte_des_cles("orders")
for client, partitions in sorted(carte.items()):
    print(client, partitions)

In [ ]:
# Verification 6.6
assert len(carte) == 7, "A revoir : 7 clients trouves"
print("OK  7 clients trouves")
assert all(len(p) == 1 for p in carte.values()), "A revoir : chaque client dans une seule partition"
print("OK  chaque client dans une seule partition")
assert carte.get("C002") == [0], "A revoir : C002 en partition 0, comme avec le producteur Java"
print("OK  C002 en partition 0, comme avec le producteur Java")

**Résultat attendu** : 7 lignes pour 7 clients, chacun dans une seule partition : `C002` en 0, `C006` et `C007` en 1, `C001`, `C003`, `C004` et `C005` en 2.

### 6.7 Ajouter des partitions

On peut augmenter le nombre de partitions d'un topic, jamais le réduire. Les messages existants restent où ils sont.

Un producteur déjà lancé garde en cache les métadonnées du topic : il ne voit les nouvelles partitions qu'au rafraîchissement suivant, jusqu'à 5 minutes plus tard (`topic.metadata.refresh.interval.ms` pour librdkafka, `metadata.max.age.ms` pour Java). Pour que le second envoi utilise les 6 partitions, recréez le producteur.

**Exemple en ligne de commande** : demande de réduction à 2 partitions.

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --alter --topic orders --partitions 2

Équivalent en ligne de commande de l'ajout et du second envoi (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --alter --topic orders --partitions 6
kafka-console-producer.sh --bootstrap-server $BS --topic orders \
  --reader-property parse.key=true --reader-property 'key.separator=|' \
  < /opt/lab/data/orders.txt
```

In [ ]:
# TODO 6.7 : passer orders a 6 partitions, renvoyer les commandes, mesurer l'effet
#   1. admin.create_partitions([NewPartitions("orders", 6)]) renvoie {nom: futur} : attendez le resultat.
#   2. Recreez producteur avec la meme configuration qu'a l'etape 6.4, puis renvoyez
#      les 30 commandes : envoyes_2 = produire_commandes().
#   3. Recalculez fins_2 = offsets_de_fin("orders") et carte_2 = carte_des_cles("orders").
#   4. Demandez le retour a 3 partitions et rangez le code d'erreur dans code_reduction.
envoyes_2, fins_2, carte_2, code_reduction = 0, {}, {}, None

print(fins_2)
for client, partitions in sorted(carte_2.items()):
    print(client, partitions)
print("code_reduction :", code_reduction)

In [ ]:
# Verification 6.7
assert envoyes_2 == 30, "A revoir : 30 messages renvoyes"
print("OK  30 messages renvoyes")
assert fins_2 == {0: 12, 1: 16, 2: 21, 3: 0, 4: 0, 5: 11}, "A revoir : offsets de fin : %s" % fins_2
print("OK  offsets de fin : %s" % fins_2)
deplaces = sorted(c for c, p in carte_2.items() if len(p) > 1)
assert deplaces == ["C001", "C003", "C004"], "A revoir : clients presents dans 2 partitions : %s" % deplaces
print("OK  clients presents dans 2 partitions : %s" % deplaces)
assert code_reduction == "INVALID_PARTITIONS", "A revoir : reduction refusee : %s" % code_reduction
print("OK  reduction refusee : %s" % code_reduction)

**Résultat attendu** : `{0: 12, 1: 16, 2: 21, 3: 0, 4: 0, 5: 11}`. Les clients `C001`, `C003` et `C004` apparaissent maintenant dans deux partitions, 2 et 5. Le retour à 3 partitions est refusé : `The topic orders currently has 6 partition(s); 3 would not be an increase.`

### 6.8 Configurer le topic `clickstream`

Une configuration de topic (`retention.ms`, `retention.bytes`...) remplace, pour ce topic seulement, la valeur du cluster. `retention.bytes` s'applique **par partition**.

**Exemple en ligne de commande** : configurations propres à `orders`, puis liste des topics « avec configurations particulières ».

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type topics --entity-name orders
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topics-with-overrides

Équivalent en ligne de commande du TODO (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic clickstream --partitions 6 --replication-factor 3
kafka-configs.sh --bootstrap-server $BS --alter --entity-type topics --entity-name clickstream \
  --add-config retention.ms=86400000,retention.bytes=104857600
kafka-configs.sh --bootstrap-server $BS --describe --entity-type topics --entity-name clickstream
kafka-configs.sh --bootstrap-server $BS --alter --entity-type topics --entity-name clickstream \
  --delete-config retention.bytes
```

In [ ]:
# TODO 6.8 : creer et configurer clickstream
#   1. Creez clickstream : 6 partitions, RF 3.
#   2. Posez retention.ms=86400000 (24 h) et retention.bytes=104857600 (100 Mo) :
#        ressource = ConfigResource("topic", "clickstream", incremental_configs=[
#            ConfigEntry("retention.ms", "86400000", incremental_operation=AlterConfigOpType.SET),
#            ...])
#        admin.incremental_alter_configs([ressource]) renvoie {ressource: futur}.
#   3. Ecrivez surcharges(topic) : dictionnaire {nom: valeur} des SEULES configurations propres au topic.
#        admin.describe_configs([ConfigResource("topic", topic)]) renvoie {ressource: futur} ;
#        le resultat est un dictionnaire {nom: entree}. entree.value est la valeur, entree.source
#        son origine : ConfigSource.DYNAMIC_TOPIC_CONFIG.value pour une configuration du topic.
#      Rangez surcharges("clickstream") dans avant_retrait.
#   4. Retirez retention.bytes : AlterConfigOpType.DELETE, avec None pour valeur.
def surcharges(topic):
    return {}


avant_retrait = {}

print("avant retrait :", avant_retrait)
print("apres retrait :", surcharges("clickstream"))

In [ ]:
# Verification 6.8
admin = AdminClient(CONF)
assert len(admin.list_topics("clickstream", timeout=10).topics["clickstream"].partitions) == 6, "A revoir : clickstream a 6 partitions"
print("OK  clickstream a 6 partitions")
assert avant_retrait == {"retention.ms": "86400000", "retention.bytes": "104857600"}, "A revoir : deux configurations posees : %s" % avant_retrait
print("OK  deux configurations posees : %s" % avant_retrait)
assert surcharges("clickstream") == {"retention.ms": "86400000"}, "A revoir : retention.bytes retiree, retention.ms gardee"
print("OK  retention.bytes retiree, retention.ms gardee")
assert surcharges("orders") == {}, "A revoir : orders n'a aucune configuration propre"
print("OK  orders n'a aucune configuration propre")

**Résultat attendu** : avant retrait, `retention.ms` et `retention.bytes` ; après, `retention.ms` seule. `orders` n'a aucune configuration propre, alors que `--topics-with-overrides` l'affiche quand même.

### 6.9 Produire sans clé

Sans clé, le producteur choisit lui-même la partition.

**Exemple en ligne de commande** : 1000 messages sans clé avec l'outil Java (cette commande est à exécuter : le `TODO` en ajoute 1000 autres avec le client Python, pour comparer).

In [ ]:
# L'outil Java attendrait 60 s par message si clickstream n'existait pas : on verifie d'abord
admin = AdminClient(CONF)
if "clickstream" in admin.list_topics(timeout=10).topics:
    !docker exec kafka-tools bash -c "seq 1 1000 | sed 's/^/click-/' | kafka-console-producer.sh --bootstrap-server $BS --topic clickstream"
    !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic clickstream
else:
    print("clickstream absent : terminez d'abord l'etape 6.8")

In [ ]:
# TODO 6.9 : envoyer 1000 messages sans cle avec le client Python, compter par partition
#   1. Envoyez "click-1" a "click-1000" dans clickstream, sans cle, avec producteur.
#   2. Comptez les messages confirmes par partition dans repartition {partition: nombre} :
#      dans le rappel de livraison, msg.partition() donne la partition.
repartition = {}

print(repartition)

In [ ]:
# Verification 6.9
assert sum(repartition.values()) == 1000, "A revoir : 1000 messages confirmes"
print("OK  1000 messages confirmes")
assert max(repartition.values()) >= 500, "A revoir : messages concentres, pas repartis un par un : %s" % repartition
print("OK  messages concentres, pas repartis un par un : %s" % repartition)

**Résultat attendu** : les 1000 messages de l'outil Java se retrouvent tous, ou presque tous, dans une seule partition. Ceux du client Python sont aussi concentrés dans une partition, pas forcément la même ; selon la vitesse du poste, de 0 à une trentaine de messages partent dans chacune des autres partitions. Les partitions choisies varient d'un poste à l'autre.

### 6.10 Kafka UI et suppression d'un topic

Ouvrez http://localhost:8080, menu « Topics », topic `orders`, onglet « Messages » : retrouvez la partition, l'offset et la clé de quelques commandes.

La suppression d'un topic est asynchrone : le contrôleur l'enregistre, puis les brokers effacent les fichiers.

**Exemple en ligne de commande** :

In [ ]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --list

Équivalent en ligne de commande du TODO (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic tmp-test --partitions 1 --replication-factor 3
kafka-topics.sh --bootstrap-server $BS --delete --topic tmp-test
```

In [ ]:
# TODO 6.10 : creer puis supprimer le topic tmp-test
#   1. Creez tmp-test (1 partition, RF 3).
#   2. Supprimez-le : admin.delete_topics(["tmp-test"]) renvoie {nom: futur}.
#   3. Avec attendre(), attendez qu'il ne figure plus dans admin.list_topics(timeout=10).topics,
#      puis rangez la liste triee des noms de topics dans topics_restants.
topics_restants = []

print(topics_restants)

In [ ]:
# Verification 6.10
assert topics_restants and "tmp-test" not in topics_restants, "A revoir : tmp-test supprime"
print("OK  tmp-test supprime")
assert {"orders", "clickstream"} <= set(topics_restants), "A revoir : orders et clickstream conserves pour le TP3"
print("OK  orders et clickstream conserves pour le TP3")

**Résultat attendu** : `__consumer_offsets`, `clickstream` et `orders`, plus `loadgen` et `lab-validate` en mode `extended`. `tmp-test` n'apparaît plus.

## Fiche des topics livrés

La cellule suivante résume les topics livrés à NovaShop. Ces topics sont réutilisés au TP3 : ne les supprimez pas.

In [ ]:
admin = AdminClient(CONF)
fiche = []
for nom in ("orders", "clickstream"):
    meta = admin.list_topics(nom, timeout=10).topics[nom]
    if meta.error is not None:
        fiche.append((nom, "absent", "", "", ""))
        continue
    rf = len(meta.partitions[0].replicas)
    configs = ", ".join("%s=%s" % kv for kv in sorted(surcharges(nom).items())) or "aucune"
    sortie = !docker exec kafka-tools kafka-get-offsets.sh --bootstrap-server $BS --topic {nom} 2>&1
    messages = sum(int(n) for n in re.findall(r"^%s:\d+:(\d+)$" % nom, sortie.n, re.M))
    fiche.append((nom, len(meta.partitions), rf, configs, messages))

tableau(fiche, ["Topic", "Partitions", "RF", "Configurations propres", "Messages"])

**Attendu** : `orders`, 6 partitions, RF 3, aucune configuration propre, 60 messages ; `clickstream`, 6 partitions, RF 3, `retention.ms=86400000`, 2000 messages (1000 par l'outil Java, 1000 par le client Python).

## 7. Questions de réflexion

1. Pourquoi désactiver la création automatique des topics en production ?
2. Le facteur de réplication peut-il dépasser le nombre de brokers ? Et le nombre de partitions ?
3. Pourquoi Kafka refuse-t-il de réduire le nombre de partitions ?
4. Après l'ajout de partitions, l'ordre des commandes du client `C001` est-il encore garanti ? Que conseilleriez-vous à l'équipe Commandes ?
5. Pourquoi 1000 messages sans clé se retrouvent-ils dans une seule partition, alors qu'on parle souvent de répartition « round-robin » ?
6. Comment choisir le nombre de partitions d'un nouveau topic ?
7. Une application Java et une application Python écrivent dans `orders` avec la même clé client. Quel réglage faut-il vérifier, et pourquoi ?